<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/TIC-TAC-TOE_lektioner/Lab_2_Facit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Facit – Tre i rad – Alla 6 lektioner

**Målgrupp:** Lärare och handledare  
**Innehåll:** Fullständiga svar och kod för alla 6 lektioner  

### Upphovspersoner
Originalversion: Mattias Tiger, mattias.tiger@liu.se  
Förenklad version för nybörjare: baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 🎮 Lektion 1 – Förstå spelet

### Nyckelbegrepp:
- Spelplanen är en 2D-lista (lista av listor)
- `0` = tom, `1` = X (spelare 1), `2` = O (spelare 2)
- 8 möjliga vinnande linjer: 3 rader, 3 kolumner, 2 diagonaler


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import copy

# ============================================================
# INSTÄLLNINGAR
# ============================================================
BOARD_SIZE = 3

# ============================================================
# FUNKTION: Skapa ett tomt bräde
# ============================================================
def skapa_tomt_brade():
    """Skapar ett nytt tomt 3x3-bräde."""
    return [[0] * BOARD_SIZE for _ in range(BOARD_SIZE)]

# ============================================================
# FUNKTION: Visa brädet i ett läsbart format
# ============================================================
def visa_brade(board):
    """Skriver ut spelplanen med symboler."""
    symboler = {0: '.', 1: 'X', 2: 'O'}
    print('Spelplanen:')
    for rad in range(BOARD_SIZE):
        rad_text = ' | '.join(symboler[board[rad][kol]] for kol in range(BOARD_SIZE))
        print(f'  {rad_text}')
    print()

# ============================================================
# FUNKTION: Kolla vinnare
# ============================================================
def check_winner(board):
    """
    Kollar om det finns en vinnare på spelplanen.

    Returnerar:
        1     → Spelare 1 (X) vinner
        2     → Spelare 2 (O) vinner
        3     → Oavgjort (alla rutor fyllda, ingen vinnare)
        None  → Spelet pågår fortfarande
    """
    # Kolla alla rader
    for rad in range(BOARD_SIZE):
        if board[rad][0] != 0 and all(board[rad][k] == board[rad][0] for k in range(BOARD_SIZE)):
            return board[rad][0]

    # Kolla alla kolumner
    for kol in range(BOARD_SIZE):
        if board[0][kol] != 0 and all(board[k][kol] == board[0][kol] for k in range(BOARD_SIZE)):
            return board[0][kol]

    # Kolla diagonalen (vänster-uppifrån → höger-nerifrån)
    if board[0][0] != 0 and all(board[k][k] == board[0][0] for k in range(BOARD_SIZE)):
        return board[0][0]

    # Kolla antidiagonalen (höger-uppifrån → vänster-nerifrån)
    if board[0][BOARD_SIZE-1] != 0 and all(board[k][BOARD_SIZE-1-k] == board[0][BOARD_SIZE-1] for k in range(BOARD_SIZE)):
        return board[0][BOARD_SIZE-1]

    # Kolla om det är oavgjort (inga tomma rutor kvar)
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 3

    return None  # Spelet pågår

print('✅ Lektion 1-funktioner laddade!')
print('\nDemo: Spelare X vinner på rad 0:')
test_brade = [[1, 1, 1], [2, 2, 0], [0, 0, 0]]
visa_brade(test_brade)
print('Vinnare:', check_winner(test_brade), '(1 = X)')

### ✏️ Svar – Lektion 1 övningar

In [ ]:
# Övning 1.1 – Vilken spelare vinner?
# SVAR:
brade_a = [[1,1,1],[2,2,0],[0,0,0]]  # Rad 0: X X X → Spelare 1 (X) vinner
brade_b = [[2,1,2],[2,1,0],[1,0,1]]  # Kolumn 1: X X X → Spelare 1 (X) vinner
brade_c = [[1,2,1],[2,2,2],[1,1,2]]  # Rad 1: O O O → Spelare 2 (O) vinner

for namn, b in [('A', brade_a), ('B', brade_b), ('C', brade_c)]:
    v = check_winner(b)
    symbol = {1:'X', 2:'O', 3:'Oavgjort', None:'Pågår'}
    print(f'Bräde {namn}: Vinnare = {symbol[v]}')
    visa_brade(b)

In [ ]:
# Övning 1.2 – Räkna tomma celler
# SVAR:
def rakna_tomma(board):
    """Räknar antalet tomma celler (0) på brädet."""
    antal = 0
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:
                antal += 1
    return antal

# Alternativt med list comprehension:
def rakna_tomma_v2(board):
    return sum(1 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE) if board[i][j] == 0)

test = [[1, 0, 2], [0, 1, 0], [2, 0, 0]]
print('Tomma celler:', rakna_tomma(test))  # Svar: 5
print('Tomma celler v2:', rakna_tomma_v2(test))

---
## 🎮 Lektion 2 – Spela spelet själv

### Nyckelbegrepp:
- `make_move()` placerar en spelare på brädet
- Kontrollera alltid att cellen är tom INNAN du placerar
- Varvande spelare: 1 → 2 → 1 → 2 ...


In [ ]:
# ============================================================
# FUNKTION: Gör ett drag
# ============================================================
def make_move(board, row, col, player):
    """
    Gör ett drag på spelplanen.

    Argument:
        board:  Spelplanen (2D-lista 3x3)
        row:    Vilken rad (0, 1 eller 2)
        col:    Vilken kolumn (0, 1 eller 2)
        player: Vem spelar (1=X, 2=O)

    Returnerar:
        True  → Draget lyckades
        False → Rutan var redan fylld
    """
    # Kontrollera att rutan är tom
    if board[row][col] != 0:
        return False  # Rutan är redan fylld!

    # Placera spelarens symbol
    board[row][col] = player
    return True  # Draget lyckades!

print('✅ make_move laddad!')

# Demo
demo = skapa_tomt_brade()
make_move(demo, 1, 1, 1)  # X spelar mitten
make_move(demo, 0, 0, 2)  # O spelar hörn
visa_brade(demo)

In [ ]:
# ============================================================
# INTERAKTIVT SPEL – Människa vs Människa
# ============================================================
spel_brade = skapa_tomt_brade()
aktuell_spelare = [1]

status_label = widgets.Label(value='🎮 Spelare X börjar! Klicka på en ruta.')

def skapa_knappar():
    knappar = []
    for rad in range(BOARD_SIZE):
        rad_knappar = []
        for kol in range(BOARD_SIZE):
            btn = widgets.Button(
                description='',
                layout=widgets.Layout(width='80px', height='80px')
            )
            btn.rad = rad
            btn.kol = kol
            btn.on_click(knapp_klickad)
            rad_knappar.append(btn)
        knappar.append(rad_knappar)
    return knappar

def knapp_klickad(btn):
    rad, kol = btn.rad, btn.kol
    spelare = aktuell_spelare[0]
    symbol = 'X' if spelare == 1 else 'O'

    if make_move(spel_brade, rad, kol, spelare):
        btn.description = symbol
        btn.style.button_color = 'lightblue' if spelare == 1 else 'lightyellow'

        vinnare = check_winner(spel_brade)
        if vinnare == 3:
            status_label.value = '🤝 Oavgjort!'
        elif vinnare:
            status_label.value = f'🏆 Spelare {symbol} vinner!'
        else:
            aktuell_spelare[0] = 2 if spelare == 1 else 1
            nasta = 'X' if aktuell_spelare[0] == 1 else 'O'
            status_label.value = f'🎮 Spelare {nasta}:s tur!'

def aterstall_spel(btn):
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            spel_brade[i][j] = 0
    aktuell_spelare[0] = 1
    status_label.value = '🎮 Spelet återstartat! Spelare X börjar.'
    for rad in spel_knappar:
        for k in rad:
            k.description = ''
            k.style.button_color = 'white'

spel_knappar = skapa_knappar()
aterstall_btn = widgets.Button(description='🔄 Nytt spel', button_style='warning')
aterstall_btn.on_click(aterstall_spel)

display(status_label)
display(widgets.VBox([widgets.HBox(rad) for rad in spel_knappar]))
display(aterstall_btn)

---
## 🎮 Lektion 3 – Slumpmässig AI-agent

### Nyckelbegrepp:
- En **agent** är ett program som tar beslut och utför handlingar
- `find_empty_cells()` hittar alla lediga rutor
- `random.choice()` väljer slumpmässigt ur en lista


In [ ]:
# ============================================================
# FUNKTION: Hitta tomma celler
# ============================================================
def find_empty_cells(board):
    """Returnerar en lista med alla tomma cellers (rad, kol)-koordinater."""
    tomma = []
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:  # 0 = tom cell
                tomma.append((rad, kol))
    return tomma

# ============================================================
# FUNKTION: Slumpmässig agent
# ============================================================
def random_agent(board):
    """Väljer ett slumpmässigt drag bland de lediga rutorna."""
    tomma = find_empty_cells(board)
    if tomma:
        return random.choice(tomma)  # Välj slumpmässigt!
    return None  # Inga lediga rutor

print('✅ Lektion 3-funktioner laddade!')

# Demo
test = [[1, 0, 2], [0, 1, 0], [2, 0, 0]]
print('Tomma celler:', find_empty_cells(test))
print('Slumpmässigt drag:', random_agent(test))

In [ ]:
# ============================================================
# SIMULERING: Slumpmässig vs Slumpmässig (100 spel)
# ============================================================
def simulera_spel(antal_spel=100):
    """Låter två slumpmässiga agenter spela mot varandra."""
    vinster = {1: 0, 2: 0, 3: 0}

    for _ in range(antal_spel):
        brade = skapa_tomt_brade()
        spelare = 1

        while True:
            drag = random_agent(brade)
            if drag is None:
                break
            make_move(brade, drag[0], drag[1], spelare)
            vinnare = check_winner(brade)
            if vinnare is not None:
                vinster[vinnare] += 1
                break
            spelare = 2 if spelare == 1 else 1

    print(f'📊 Resultat efter {antal_spel} spel:')
    print(f'  X vinner: {vinster[1]} ({vinster[1]/antal_spel*100:.0f}%)')
    print(f'  O vinner: {vinster[2]} ({vinster[2]/antal_spel*100:.0f}%)')
    print(f'  Oavgjort: {vinster[3]} ({vinster[3]/antal_spel*100:.0f}%)')

simulera_spel(200)

---
## 🎮 Lektion 4 – Din egen regelbaserade agent

### Nyckelbegrepp:
- **Regelbaserad AI** följer explicita if-satser
- Regel 1: Vinn om möjligt
- Regel 2: Blockera motståndarens vinst
- Regel 3: Ta mitten
- Regel 4 (bonus): Ta ett hörn


In [ ]:
# ============================================================
# FUNKTION: Hitta vinnardrag
# ============================================================
def hitta_vinnardrag(board, spelare):
    """
    Kollar om spelaren kan vinna med nästa drag.
    Testar alla tomma celler och kollar om det leder till vinst.
    """
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:  # Tom cell
                board[rad][kol] = spelare  # Prova draget
                if check_winner(board) == spelare:
                    board[rad][kol] = 0  # Ångra
                    return (rad, kol)    # Returera vinnardrag!
                board[rad][kol] = 0      # Ångra
    return None

# ============================================================
# FUNKTION: Hitta blockdrag
# ============================================================
def hitta_blockdrag(board, spelare):
    """Kollar om vi måste blockera motståndarens vinnardrag."""
    motstandare = 2 if spelare == 1 else 1
    return hitta_vinnardrag(board, motstandare)

# ============================================================
# FUNKTION: Ta mitten
# ============================================================
def ta_mitten(board):
    """Tar mittcellen om den är ledig."""
    if board[1][1] == 0:
        return (1, 1)
    return None

# ============================================================
# FUNKTION: Ta ett hörn
# ============================================================
def ta_horn(board):
    """Tar ett ledigt hörn om möjligt."""
    horn = [(0,0), (0,2), (2,0), (2,2)]
    for (r, k) in horn:
        if board[r][k] == 0:
            return (r, k)
    return None

# ============================================================
# FUNKTION: Smart agent (kombinerar alla regler)
# ============================================================
def smart_agent(board, spelare):
    """
    Intelligent agent med fyra prioriterade regler.
    Regel 1: Vinn om möjligt
    Regel 2: Blockera motspelaren
    Regel 3: Ta mitten
    Regel 4: Ta ett hörn
    """
    drag = hitta_vinnardrag(board, spelare)
    if drag: return drag

    drag = hitta_blockdrag(board, spelare)
    if drag: return drag

    drag = ta_mitten(board)
    if drag: return drag

    drag = ta_horn(board)
    if drag: return drag

    # Annars: slumpmässigt drag
    tomma = find_empty_cells(board)
    return random.choice(tomma) if tomma else None

print('✅ Lektion 4-funktioner laddade!')

---
## 🎮 Lektion 5 – Sökalgoritmerna (DFS & BFS)

### Nyckelbegrepp:
- **DFS** (Djupet-först): Går djupt i trädet innan den backar
- **BFS** (Bredden-först): Utforskar alla grannar på en nivå innan nästa
- Båda bygger ett **spelträd** av möjliga framtida drag


In [ ]:
from collections import deque

# ============================================================
# FUNKTION: DFS-agent
# ============================================================
def dfs_agent(board, spelare):
    """
    DFS-agent: Letar igenom spelträdet på djupet.
    Returnerar det bästa draget för spelaren.
    """
    def dfs(brade, akt_spelare, djup=0):
        vinnare = check_winner(brade)
        if vinnare == spelare:
            return 1, None   # Vi vinner!
        elif vinnare and vinnare != spelare:
            return -1, None  # Vi förlorar!
        elif vinnare == 3:
            return 0, None   # Oavgjort

        tomma = find_empty_cells(brade)
        if not tomma:
            return 0, None

        basta_poang = -2
        basta_drag = tomma[0]

        for (rad, kol) in tomma:
            brade[rad][kol] = akt_spelare
            nasta = 2 if akt_spelare == 1 else 1
            poang, _ = dfs(brade, nasta, djup + 1)
            poang = -poang  # Invertera för motspelaren
            brade[rad][kol] = 0

            if poang > basta_poang:
                basta_poang = poang
                basta_drag = (rad, kol)

        return basta_poang, basta_drag

    _, drag = dfs(board, spelare)
    return drag

# ============================================================
# FUNKTION: BFS-agent
# ============================================================
def bfs_agent(board, spelare):
    """
    BFS-agent: Utforskar alla möjliga drag nivå för nivå.
    Hittar vinst på kortast möjliga väg.
    """
    ko = deque()

    for (rad, kol) in find_empty_cells(board):
        nytt = copy.deepcopy(board)
        nytt[rad][kol] = spelare
        ko.append((nytt, (rad, kol), spelare))

    basta_drag = find_empty_cells(board)
    basta_drag = basta_drag[0] if basta_drag else None

    while ko:
        akt_brade, forsta_drag, akt_spelare = ko.popleft()
        vinnare = check_winner(akt_brade)

        if vinnare == spelare:
            return forsta_drag
        elif vinnare is not None:
            continue

        nasta = 2 if akt_spelare == 1 else 1
        for (rad, kol) in find_empty_cells(akt_brade):
            nytt = copy.deepcopy(akt_brade)
            nytt[rad][kol] = nasta
            ko.append((nytt, forsta_drag, nasta))

    return basta_drag

print('✅ Lektion 5-funktioner laddade!')

---
## 🎮 Lektion 6 – Minimax: Perfekt AI

### Nyckelbegrepp:
- **Minimax** tänker som BÅDA spelarna
- **MAX**-nivå: Vi väljer draget med HÖGST poäng
- **MIN**-nivå: Motståndaren väljer draget med LÄGST poäng
- Poäng: +10 (vi vinner), -10 (vi förlorar), 0 (oavgjort)
- Minimax-agenten kan **aldrig förlora** vid Tre i rad!


In [ ]:
# ============================================================
# FUNKTION: Utvärdera brädet
# ============================================================
def evaluate_board(board, spelare):
    """
    Utvärderar spelplanen och ger en poäng.
    +10 = bra (vi vinner), -10 = dåligt (vi förlorar), 0 = oavgjort/pågår
    """
    vinnare = check_winner(board)
    if vinnare == spelare:
        return 10
    elif vinnare == 3:
        return 0
    elif vinnare is not None:
        return -10
    return 0

# ============================================================
# FUNKTION: Minimax
# ============================================================
def minimax(board, djup, maximerar, spelare):
    """
    Minimax-algoritmen: Väljer det optimala draget.

    Argument:
        board:      Aktuell spelplan
        djup:       Sökdjup (ökar för varje rekursivt anrop)
        maximerar:  True = vår tur (MAX), False = motståndarens tur (MIN)
        spelare:    Vi (1 eller 2)
    """
    motstandare = 2 if spelare == 1 else 1

    # Beräkna poäng (minus djup = föredrar kortare vägar till vinst)
    poang = evaluate_board(board, spelare)
    if poang == 10:
        return 10 - djup
    if poang == -10:
        return -10 + djup

    # Kolla om det är oavgjort
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 0

    if maximerar:
        # Vår tur: välj draget med HÖGST poäng
        basta = -100
        for (rad, kol) in find_empty_cells(board):
            board[rad][kol] = spelare
            basta = max(basta, minimax(board, djup + 1, False, spelare))
            board[rad][kol] = 0
        return basta
    else:
        # Motståndarens tur: väljer draget med LÄGST poäng
        basta = 100
        for (rad, kol) in find_empty_cells(board):
            board[rad][kol] = motstandare
            basta = min(basta, minimax(board, djup + 1, True, spelare))
            board[rad][kol] = 0
        return basta

def minimax_agent(board, spelare):
    """Väljer det bästa draget med Minimax."""
    basta_poang = -100
    basta_drag = None

    for (rad, kol) in find_empty_cells(board):
        board[rad][kol] = spelare
        poang = minimax(board, 0, False, spelare)
        board[rad][kol] = 0

        if poang > basta_poang:
            basta_poang = poang
            basta_drag = (rad, kol)

    return basta_drag

print('✅ Lektion 6-funktioner laddade!')
print('Testar minimax_agent på tomt bräde...')
tomt = skapa_tomt_brade()
drag = minimax_agent(tomt, 1)
print(f'Minimax väljer drag: {drag}  (borde vara mitten (1,1) eller hörn)')

In [ ]:
# ============================================================
# JÄMFÖRELSE: Alla agenter mot slumpmässig (50 spel vardera)
# ============================================================
def testa_agent(agent_func, agent_namn, antal=50):
    """Låter en agent spela mot den slumpmässiga agenten."""
    vinster = {1: 0, 2: 0, 3: 0}

    for _ in range(antal):
        brade = skapa_tomt_brade()
        spelare = 1

        while True:
            if spelare == 1:
                drag = agent_func(brade, 1)
            else:
                drag = random_agent(brade)

            if drag is None: break
            make_move(brade, drag[0], drag[1], spelare)
            vinnare = check_winner(brade)
            if vinnare is not None:
                vinster[vinnare] += 1
                break
            spelare = 2 if spelare == 1 else 1

    print(f'Agent: {agent_namn:20s} | Vinster: {vinster[1]:3d}/{antal}  ({vinster[1]/antal*100:5.1f}%)')

print('📊 Agentjämförelse (varje agent spelar mot slumpmässig, 50 spel):')
print('-' * 60)
testa_agent(lambda b, p: random_agent(b), 'Slumpmässig')
testa_agent(smart_agent,                  'Regelbaserad')
testa_agent(dfs_agent,                    'DFS')
testa_agent(bfs_agent,                    'BFS')
testa_agent(minimax_agent,                'Minimax')

---
## 🎉 Sammanfattning – Alla 6 lektioner

| Lektion | Agent | Strategi | Styrka | Svaghet |
|---------|-------|----------|--------|---------|
| 1 | — | Förstå spelplanen | Grundläggande logik | Inget beslut |
| 2 | Människa | Klicka knappar | Kul att spela | Långsam |
| 3 | Slumpmässig | `random.choice()` | Enkel, oförutsägbar | Ingen strategi |
| 4 | Regelbaserad | If-satser | Snabb, vinner ofta | Begränsade regler |
| 5 | DFS/BFS | Sökträd | Ser framåt | Långsam, ingen MIN |
| 6 | Minimax | MAX+MIN | Perfekt spel | Långsam för stora spel |

### Vad har vi lärt oss?
- 🔢 **Brädet** representeras som en 2D-lista
- 🏆 **check_winner()** kollar 8 vinnande linjer
- 🎲 **Slumpmässig agent** väljer utan strategi
- 🧠 **Regelbaserad agent** följer explicita if-satser
- 🔍 **DFS/BFS** bygger ett spelträd och letar efter vinst
- ♟️ **Minimax** tänker som båda spelarna → kan aldrig förlora!
